# Augmentation Visual Check

This notebook renders random augmented samples from your dataset so you can
visually verify that polygons track the image correctly through every
transform in `src/augmentation.py`.

Just edit `CONFIG` below and run all.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# allow running from repo root OR from notebooks/
REPO_ROOT = Path.cwd()
if (REPO_ROOT.name == 'notebooks'):
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print('repo:', REPO_ROOT)

In [ ]:
# ---------------- CONFIG ----------------
CONFIG = {
    'dataset_root': '/mnt/DATA2/seilov/hw_dataset/hw_dataset',
    'labels_txt':   str(REPO_ROOT / 'labels' / 'labels.txt'),
    'min_score':    0.5,
    'image_size':   640,
    'tier':         'handwriting',   # 'none' | 'standard' | 'handwriting'
    'num_samples':  8,               # how many images to sample
    'num_augs':     4,               # how many aug variants per image
    'seed':         123,
}

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.augmentation import Augmenter, AugConfig
from src.dataset import parse_labels_txt
from src.target_gen import TargetConfig, generate_targets
from src.utils import denormalize, draw_polygons

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

In [ ]:
items = parse_labels_txt(Path(CONFIG['labels_txt']))
# pick samples with at least a few boxes above score threshold
items_with_boxes = []
for rel, boxes in items:
    kept = [b for b in boxes if (b.get('score') or 1.0) >= CONFIG['min_score']]
    if len(kept) >= 3:
        items_with_boxes.append((rel, kept))
print('usable images:', len(items_with_boxes))

picked = random.sample(items_with_boxes, min(CONFIG['num_samples'], len(items_with_boxes)))
print('picked', len(picked), 'samples')

In [ ]:
aug_cfg = AugConfig(tier=CONFIG['tier'], image_size=CONFIG['image_size'])
train_aug = Augmenter(aug_cfg, train=True)
val_aug = Augmenter(AugConfig(tier='none', image_size=CONFIG['image_size']), train=False)

def load_image(rel):
    p = Path(CONFIG['dataset_root']) / rel
    return np.array(Image.open(p).convert('RGB'))

def boxes_to_polys(boxes):
    return [np.asarray(b['points'], dtype=np.float32).reshape(-1, 2) for b in boxes]

## 1) Raw image vs augmented variants
Polygons overlayed. Column 0 = no aug (val pipeline), columns 1..K = training augs.

In [ ]:
cols = CONFIG['num_augs'] + 1
rows = len(picked)
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if rows == 1:
    axes = axes[None, :]

for r, (rel, boxes) in enumerate(picked):
    img = load_image(rel)
    polys = boxes_to_polys(boxes)

    # column 0: val (resize+pad only)
    img_v, polys_v, _ = val_aug(img, polys)
    img_v_u8 = denormalize(np.transpose(img_v, (2, 0, 1))) if img_v.dtype != np.uint8 else img_v
    # val pipeline also normalizes -> reverse for display
    img_v_u8 = denormalize(np.transpose(img_v, (2, 0, 1)).astype(np.float32) if img_v.dtype != np.uint8 else img_v)
    axes[r, 0].imshow(draw_polygons(img_v_u8, polys_v))
    axes[r, 0].set_title(f'val | {len(polys_v)} boxes', fontsize=9)
    axes[r, 0].axis('off')

    for c in range(CONFIG['num_augs']):
        img_t, polys_t, _ = train_aug(img, polys)
        img_t_u8 = denormalize(np.transpose(img_t, (2, 0, 1)).astype(np.float32))
        axes[r, c + 1].imshow(draw_polygons(img_t_u8, polys_t))
        axes[r, c + 1].set_title(f'aug #{c + 1} | {len(polys_t)} boxes', fontsize=9)
        axes[r, c + 1].axis('off')

plt.tight_layout()
plt.show()

## 2) Targets: probability / threshold / mask maps
On a single augmented sample, visualize the DBNet++ training targets.

In [ ]:
rel, boxes = picked[0]
img = load_image(rel)
polys = boxes_to_polys(boxes)

img_t, polys_t, _ = train_aug(img, polys)
img_u8 = denormalize(np.transpose(img_t, (2, 0, 1)).astype(np.float32))

tgt_cfg = TargetConfig(shrink_ratio=0.4, thresh_min=0.3, thresh_max=0.7)
tgts = generate_targets(image_shape=img_u8.shape[:2], polygons=polys_t, cfg=tgt_cfg)

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
axes[0].imshow(draw_polygons(img_u8, polys_t)); axes[0].set_title('image + polys'); axes[0].axis('off')
axes[1].imshow(tgts['prob_map'], cmap='hot', vmin=0, vmax=1);   axes[1].set_title('prob_map (shrunk)'); axes[1].axis('off')
axes[2].imshow(tgts['prob_mask'], cmap='gray', vmin=0, vmax=1); axes[2].set_title('prob_mask'); axes[2].axis('off')
axes[3].imshow(tgts['thresh_map'], cmap='hot', vmin=0, vmax=1); axes[3].set_title('thresh_map'); axes[3].axis('off')
axes[4].imshow(tgts['thresh_mask'], cmap='gray', vmin=0, vmax=1); axes[4].set_title('thresh_mask (ring)'); axes[4].axis('off')
plt.tight_layout(); plt.show()

## 3) Tier comparison
Same image, all three tiers side-by-side.

In [ ]:
tiers = ['none', 'standard', 'handwriting']
rel, boxes = picked[min(1, len(picked) - 1)]
img = load_image(rel)
polys = boxes_to_polys(boxes)

fig, axes = plt.subplots(1, len(tiers), figsize=(6 * len(tiers), 6))
for ax, tier in zip(axes, tiers):
    aug = Augmenter(AugConfig(tier=tier, image_size=CONFIG['image_size']), train=tier != 'none')
    img_t, polys_t, _ = aug(img, polys)
    img_u8 = denormalize(np.transpose(img_t, (2, 0, 1)).astype(np.float32))
    ax.imshow(draw_polygons(img_u8, polys_t))
    ax.set_title(f'tier = {tier} | {len(polys_t)} boxes')
    ax.axis('off')
plt.tight_layout(); plt.show()